# EDA on Jobtech API data

In [1]:
import requests
import json

url = 'https://jobsearch.api.jobtechdev.se'
url_for_search = f"{url}/search"


def _get_ads(params):
    headers = {'accept': 'application/json'}
    response = requests.get(url_for_search, headers=headers, params=params)
    response.raise_for_status()  # check for http errors
    return json.loads(response.content.decode('utf8'))


def example_search_return_number_of_hits(query):
    # limit: 0 means no ads, just a value of how many ads were found.
    search_params = {'q': query, 'limit': 0}
    json_response = _get_ads(search_params)
    number_of_hits = json_response['total']['value']
    print(f"\nNumber of hits = {number_of_hits}")


def example_search_loop_through_hits(query):
    # limit = 100 is the max number of hits that can be returned.
    # If there are more (which you find with ['total']['value'] in the json response)
    # you have to use offset and multiple requests to get all ads.
    search_params = {'q': query, 'limit': 100}
    json_response = _get_ads(search_params)
    hits = json_response['hits']
    for hit in hits:   
        print(f"{hit['headline']}, {hit['employer']['name']}")


query = 'lärare uppsala'
example_search_loop_through_hits(query)



Lärare Sh och Re - Uppsala yrkesgymnasium Jälla (vikariat), UPPSALA KOMMUN
Är du vår nya idrottslärare?, Gluntens Montessoriskola AB
Vill du arbeta med våra elever i åk 6-9?, Gluntens Montessoriskola AB
Vill du arbeta med våre elever i åk 6-9?, Gluntens Montessoriskola AB
Lärare åk 4-6, Gluntens Montessoriskola AB
Lärare i matematik och fysik, Amerikanska Gymnasiet i Sverige AB
Lärare i engelska 30%, Amerikanska Gymnasiet i Sverige AB
Lärare till lågstadiet - Grillbyskolan, ENKÖPINGS KOMMUN
Gymnasielärare i svenska och engelska till Uppsala yrkesgymnasium Jälla, UPPSALA KOMMUN
Idrottslärare till särskildundervisningsgrupp till Sverkersskolan, UPPSALA KOMMUN
Lärare i engelska och/eller spanska, Amerikanska Gymnasiet i Sverige AB
Lärare i spanska, Raoul Wallenbergskolorna AB
Gymnasielärare i idrott Grillska Gymnasiet Uppsala, STADSMISSIONENS SKOLSTIFTELSE
Lärare i musik och fritidshem - Danmarks skola, UPPSALA KOMMUN
Lärare i NK och Ma (eller i komb. med annat ämne) - Ellen Fries Gymnasi

In [2]:
example_search_return_number_of_hits(query)


Number of hits = 25


In [3]:
search_params = {'q': query, 'limit': 100}
json_response = _get_ads(search_params)
# json_response

In [4]:
json_response.keys()

dict_keys(['total', 'positions', 'query_time_in_millis', 'result_time_in_millis', 'stats', 'freetext_concepts', 'hits'])

In [5]:
len(json_response["hits"])

25

In [6]:
json_response["hits"][1].keys()

dict_keys(['relevance', 'id', 'external_id', 'original_id', 'label', 'webpage_url', 'logo_url', 'headline', 'application_deadline', 'number_of_vacancies', 'description', 'employment_type', 'salary_type', 'salary_description', 'duration', 'working_hours_type', 'scope_of_work', 'access', 'employer', 'application_details', 'experience_required', 'access_to_own_car', 'driving_license_required', 'driving_license', 'occupation', 'occupation_group', 'occupation_field', 'workplace_address', 'must_have', 'nice_to_have', 'application_contacts', 'publication_date', 'last_publication_date', 'removed', 'removed_date', 'source_type', 'timestamp'])

In [7]:
json_response["hits"][1]["headline"]

'Lärare i matematik och fysik'

In [8]:
json_response = _get_ads({"q": "data engineer", "limit": 100})
# json_response

In [9]:
json_response["hits"][-1]["headline"], json_response["hits"][-1]["employer"]["name"]

('Machine Learning Engineer / Data Scientist', '2MNordic IT Consulting AB')

# Understand pagination of Jobtech API
- pagination reduces the load on the server and the client by providing a subset of data at a time
- with limit-offset pagination, the client can specify the number of records to be retrieved (```limit```) and the starting point (```offset```) 

In [77]:
import pandas as pd
import requests
import json

url = "https://jobsearch.api.jobtechdev.se"
url_for_search = f"{url}/search"
params = {"q": "", 
          "limit": 100,
          "occupation-field": "6Hq3_tKo_V57"}

In [78]:
offset = 0

all_jobs = []
page_params = dict(params, offset=offset)
data = _get_ads(url_for_search, page_params)
for ad in data["hits"]:
    all_jobs.append(ad)

df_offset0 = pd.DataFrame(all_jobs)
#df_offset0["id"].nunique()
df_offset0["occupation_field"]
df_offset0["concept_id"] = df_offset0["occupation_field"].apply(lambda x: x.get("concept_id") if isinstance(x, dict) else None)


In [79]:
df_offset0["concept_id"].value_counts()

concept_id
6Hq3_tKo_V57    94
bh3H_Y3h_5eD     2
kJeN_wmw_9wX     2
X82t_awd_Qyc     2
Name: count, dtype: int64

In [81]:
df_offset0[df_offset0["concept_id"]!="6Hq3_tKo_V57"]["occupation_group"]

4     {'concept_id': 'pf1K_PTz_frm', 'label': 'Produ...
34    {'concept_id': 'wNrt_Ysj_WuT', 'label': 'Biome...
41    {'concept_id': 'GMDo_DVo_Yzh', 'label': 'Lantm...
81    {'concept_id': 'pf1K_PTz_frm', 'label': 'Produ...
83    {'concept_id': 'GMDo_DVo_Yzh', 'label': 'Lantm...
92    {'concept_id': 'wNrt_Ysj_WuT', 'label': 'Biome...
Name: occupation_group, dtype: object

In [43]:
offset = 100

all_jobs = []
page_params = dict(params, offset=offset)
data = _get_ads(url_for_search, page_params)
for ad in data["hits"]:
    all_jobs.append(ad)

df_offset100 = pd.DataFrame(all_jobs)
df_offset100["id"].nunique()

100

In [44]:
print(f"The number of unique job ads from these two dataframes: {len(set(df_offset0["id"]) | set(df_offset100["id"]))}")

The number of unique job ads from these two dataframes: 200
